# 🧪 Sesión 14 · Hashing Seguro de Contraseñas (Bcrypt / Argon2)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué nunca se guarda una contraseña tal cual?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/registro  {"usuario":"ana","password":"secreta"}
HTTP/1.1 201 Created
{"usuario":"ana"}

POST /api/login  {"usuario":"ana","password":"secreta"}
HTTP/1.1 200 OK
{"usuario":"ana","autenticado":true}

POST /api/login  {"usuario":"ana","password":"incorrecta"}
HTTP/1.1 401 Unauthorized

La respuesta NUNCA contiene "password" ni el hash.
```

_El hash se guarda en el servidor; la contraseña original no se almacena en ningún lado._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Guardar contraseñas con bcrypt y salt.
- Verificar una contraseña sin descifrarla.
- Comprobar que la API nunca devuelve el hash ni la contraseña.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El registro responde sin filtrar la contraseña → HTTP 201
- [ ] El login correcto autentica → HTTP 200
- [ ] La contraseña incorrecta se rechaza → HTTP 401
- [ ] Se usa un algoritmo de hashing para contraseñas

## 🧩 El reto

Implementa registro y login con hashing seguro. La contraseña nunca debe viajar de vuelta al cliente.

**Requisitos del contrato**

- POST /api/registro recibe usuario y password, guarda solo el hash y responde 201.
- La respuesta de registro no incluye password ni hash.
- POST /api/login compara la contraseña con el hash y responde 200 o 401.
- Debes usar bcrypt (o passlib) para el hash.
- Un usuario repetido no debe sobrescribir la contraseña existente.

> **Pista conceptual:** No compares hashes con `==` si puedes usar la función de verificación del algoritmo. Bcrypt ya incluye el salt dentro del hash.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion14.md` explicando por qué no se puede «deshashear».


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [4]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:

%%writefile app.py

from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field, ConfigDict
import bcrypt

app = FastAPI(title="Sistema de Registro y Login Seguro")

usuarios_db = {}


class UsuarioEntrada(BaseModel):
    usuario: str = Field(min_length=1)
    password: str = Field(min_length=1)

    model_config = ConfigDict(str_strip_whitespace=True)


class RegistroSalida(BaseModel):
    usuario: str


class LoginSalida(BaseModel):
    usuario: str
    autenticado: bool


@app.post(
    "/api/registro",
    response_model=RegistroSalida,
    status_code=status.HTTP_201_CREATED
)
def registrar_usuario(datos: UsuarioEntrada):

    if datos.usuario in usuarios_db:
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail="El usuario ya existe"
        )

    password_hash = bcrypt.hashpw(
        datos.password.encode("utf-8"),
        bcrypt.gensalt()
    )

    usuarios_db[datos.usuario] = password_hash.decode("utf-8")

    return RegistroSalida(usuario=datos.usuario)


@app.post(
    "/api/login",
    response_model=LoginSalida,
    status_code=status.HTTP_200_OK
)
def iniciar_sesion(datos: UsuarioEntrada):

    password_hash = usuarios_db.get(datos.usuario)

    if password_hash is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Credenciales incorrectas"
        )

    password_correcta = bcrypt.checkpw(
        datos.password.encode("utf-8"),
        password_hash.encode("utf-8")
    )

    if not password_correcta:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Credenciales incorrectas"
        )

    return LoginSalida(
        usuario=datos.usuario,
        autenticado=True
    )


In [5]:


%%writefile app.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, ConfigDict
import bcrypt

app = FastAPI(title="Sistema de Registro y Login Seguro")

usuarios_db = {}

class UsuarioEntrada(BaseModel):
    usuario: str = Field(min_length=1)
    password: str = Field(min_length=1)

    model_config = ConfigDict(str_strip_whitespace=True)

class RegistroSalida(BaseModel):
    usuario: str

class LoginSalida(BaseModel):
    usuario: str
    autenticado: bool

@app.post("/api/registro", response_model=RegistroSalida, status_code=201)
def registrar_usuario(datos: UsuarioEntrada):

    if datos.usuario in usuarios_db:
        raise HTTPException(
            status_code=409,
            detail="El usuario ya existe"
        )

    password_hash = bcrypt.hashpw(
        datos.password.encode("utf-8"),
        bcrypt.gensalt()
    )

    usuarios_db[datos.usuario] = password_hash.decode("utf-8")

    return RegistroSalida(usuario=datos.usuario)


@app.post("/api/login", response_model=LoginSalida, status_code=200)
def iniciar_sesion(datos: UsuarioEntrada):

    password_hash = usuarios_db.get(datos.usuario)

    if password_hash is None:
        raise HTTPException(
            status_code=401,
            detail="Credenciales incorrectas"
        )

    password_correcta = bcrypt.checkpw(
        datos.password.encode("utf-8"),
        password_hash.encode("utf-8")
    )

    if not password_correcta:
        raise HTTPException(
            status_code=401,
            detail="Credenciales incorrectas"
        )

    return LoginSalida(
        usuario=datos.usuario,
        autenticado=True
    )


Writing app.py


In [6]:

import os
from fastapi.testclient import TestClient
from app import app

print("ARCHIVO APP.PY:", os.path.exists("app.py"))

cliente = TestClient(app)

registro = cliente.post(
    "/api/registro",
    json={"usuario": "ana", "password": "secreta"}
)

login_correcto = cliente.post(
    "/api/login",
    json={"usuario": "ana", "password": "secreta"}
)

login_incorrecto = cliente.post(
    "/api/login",
    json={"usuario": "ana", "password": "incorrecta"}
)

usuario_repetido = cliente.post(
    "/api/registro",
    json={"usuario": "ana", "password": "otra"}
)

print("REGISTRO:", registro.status_code, registro.json())
print("LOGIN CORRECTO:", login_correcto.status_code, login_correcto.json())
print("LOGIN INCORRECTO:", login_incorrecto.status_code)
print("USUARIO REPETIDO:", usuario_repetido.status_code)

assert registro.status_code == 201
assert login_correcto.status_code == 200
assert login_incorrecto.status_code == 401
assert usuario_repetido.status_code == 409

assert "password" not in registro.json()
assert "hash" not in registro.json()
assert "password" not in login_correcto.json()
assert "hash" not in login_correcto.json()

print("\nTODAS LAS PRUEBAS CORRECTAS")


ARCHIVO APP.PY: True
REGISTRO: 201 {'usuario': 'ana'}
LOGIN CORRECTO: 200 {'usuario': 'ana', 'autenticado': True}
LOGIN INCORRECTO: 401
USUARIO REPETIDO: 409

TODAS LAS PRUEBAS CORRECTAS


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Se trabajará con Python, FastAPI y Pydantic en Google Colab. Se utilizará la biblioteca bcrypt para proteger las contraseñas mediante hashing seguro |
| **Rol** | La IA debe asumir el papel de un desarrollador backend especializado en crear API REST seguras, utilizando buenas prácticas para proteger los datos de los usuarios. |
| **Especificación** | Crear las rutas POST /api/registro y POST /api/login. El registro debe recibir usuario y contraseña, generar un hash con bcrypt y guardar únicamente el hash. El login debe verificar la contraseña. Debe responder HTTP 201 al registrar, HTTP 200 al iniciar sesión correctamente y HTTP 401 cuando las credenciales sean incorrectas. También debe evitar usuarios duplicados y nunca mostrar la contraseña ni el hash en las respuestas. |
| **Output** | Un archivo app.py funcional con las rutas de registro e inicio de sesión, utilizando bcrypt, respuestas en formato JSON y códigos HTTP correspondientes. Además, una bitácora auditorias/auditoria_sesion14.md con la explicación de la implementación y las reflexiones de la actividad. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [7]:
from verificar import probar_http

casos = [{'descripcion': 'El registro responde sin filtrar la contraseña',
  'metodo': 'POST',
  'ruta': '/api/registro',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 201,
  'json_contiene': {'usuario': 'ana'},
  'no_contiene': ['password', 'hash']},
 {'descripcion': 'El login correcto autentica',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana', 'autenticado': True}},
 {'descripcion': 'La contraseña incorrecta se rechaza',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'incorrecta'},
  'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 3 casos · app.py
✅ 01. El registro responde sin filtrar la contraseña
✅ 02. El login correcto autentica
✅ 03. La contraseña incorrecta se rechaza
------------------------------------------------------------------------------
RESULTADO: 3/3 casos correctos. ¡Resultado logrado!


In [8]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["bcrypt|passlib"], descripcion="Se usa un algoritmo de hashing para contraseñas"))
resumen(resultados)


✅ Se usa un algoritmo de hashing para contraseñas
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion14.md` a `auditorias/auditoria_sesion14.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue lograr que el sistema pudiera verificar la contraseña sin tener que conocer el hash original. Al principio no comprendía cómo podía comprobarse, pero después aprendí que bcrypt utiliza una función llamada checkpw() que permite verificar si la contraseña es correcta.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí agregar una validación para evitar que un usuario pudiera registrarse más de una vez. También utilicé modelos de respuesta para asegurarme de que la API solamente mostrara el nombre del usuario y nunca la contraseña ni el hash.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   Al principio tuve un problema porque el verificador no encontraba el archivo app.py. Esto ocurrió porque no había creado correctamente el archivo antes de ejecutar las pruebas. Después volví a ejecutar la celda de implementación y comprobé que el archivo existiera.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
